In [2]:
import argparse
import os
import time

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
import numpy as np

# ─────────────────────────────────────────────
#  Hyperparameters
# ─────────────────────────────────────────────
LATENT_DIM   = 100      # dimension of noise vector z
IMAGE_SIZE   = 64       # internal processing size (upsampled from 28×28)
CHANNELS     = 1        # grayscale
NGF          = 64       # generator feature map base size
NDF          = 64       # discriminator feature map base size
BATCH_SIZE   = 128
LR           = 0.0002
BETA1        = 0.5      # Adam β₁ (DCGAN paper recommendation)
BETA2        = 0.999
NUM_EPOCHS   = 20
SAVE_INTERVAL = 5       # save sample grid every N epochs
OUTPUT_DIR   = "dcgan_output"


# ─────────────────────────────────────────────
#  Weight Initialisation (from DCGAN paper)
#  Conv/ConvTranspose: mean=0, std=0.02
#  BatchNorm: weight~N(1,0.02), bias=0
# ─────────────────────────────────────────────
def weights_init(m):
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


# ─────────────────────────────────────────────
#  Generator
#  Input : z  ∈ R^{LATENT_DIM}
#  Output: x̂  ∈ [-1, 1]^{1×64×64}
#
#  z → FC(project & reshape) → [1024×4×4]
#    → ConvT(512, k=4, s=2, p=1) + BN + ReLU   → [512×8×8]
#    → ConvT(256, k=4, s=2, p=1) + BN + ReLU   → [256×16×16]
#    → ConvT(128, k=4, s=2, p=1) + BN + ReLU   → [128×32×32]
#    → ConvT(  1, k=4, s=2, p=1) + Tanh        → [  1×64×64]
# ─────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, ngf=NGF, channels=CHANNELS):
        super().__init__()
        self.net = nn.Sequential(
            # ── Block 0: project noise vector into spatial feature map ──
            nn.ConvTranspose2d(latent_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # ── Block 1: 4×4 → 8×8 ──
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # ── Block 2: 8×8 → 16×16 ──
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # ── Block 3: 16×16 → 32×32 ──
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # ── Block 4: 32×32 → 64×64, output layer ──
            nn.ConvTranspose2d(ngf, channels, 4, 2, 1, bias=False),
            nn.Tanh(),          # output in [-1, 1]
        )

    def forward(self, z):
        # z shape: (B, latent_dim, 1, 1)
        return self.net(z)


# ─────────────────────────────────────────────
#  Discriminator
#  Input : x  ∈ [-1, 1]^{1×64×64}
#  Output: scalar ∈ (0,1)  — P(real)
#
#  x → Conv(64,  k=4, s=2, p=1) + LeakyReLU(0.2)  → [64×32×32]
#    → Conv(128, k=4, s=2, p=1) + BN + LeakyReLU   → [128×16×16]
#    → Conv(256, k=4, s=2, p=1) + BN + LeakyReLU   → [256×8×8]
#    → Conv(512, k=4, s=2, p=1) + BN + LeakyReLU   → [512×4×4]
#    → Conv(  1, k=4, s=1, p=0) + Sigmoid          → [1×1×1]
# ─────────────────────────────────────────────
class Discriminator(nn.Module):
    def __init__(self, ndf=NDF, channels=CHANNELS):
        super().__init__()
        self.net = nn.Sequential(
            # ── Block 0: no BN in first layer (DCGAN paper) ──
            nn.Conv2d(channels, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # ── Block 1 ──
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # ── Block 2 ──
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # ── Block 3 ──
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # ── Output: 4×4 → 1×1 ──
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).view(-1)   # flatten to (B,)


# ─────────────────────────────────────────────
#  Data Loading
# ─────────────────────────────────────────────
def get_dataloader(batch_size=BATCH_SIZE):
    """
    Loads MNIST, resizes to 64×64 so the ConvTranspose stack
    matches the discriminator's receptive field exactly.
    Normalises to [-1, 1] to match the Generator's Tanh output.
    """
    transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),   # [0,1] → [-1,1]
    ])
    dataset = torchvision.datasets.MNIST(
        root="./data", train=True, download=True, transform=transform
    )
    return torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True
    )


# ─────────────────────────────────────────────
#  Loss & Label Smoothing
#  Using one-sided label smoothing (real=0.9, fake=0.0)
#  improves training stability.
# ─────────────────────────────────────────────
criterion = nn.BCELoss()

def real_label(size, device, smoothing=0.9):
    return torch.full((size,), smoothing, device=device)

def fake_label(size, device):
    return torch.zeros(size, device=device)


# ─────────────────────────────────────────────
#  Training Loop
# ─────────────────────────────────────────────
def train(args):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else
        "mps"  if torch.backends.mps.is_available() else
        "cpu"
    )
    print(f"[Device] {device}")

    dataloader = get_dataloader(BATCH_SIZE)

    # ── Models ──
    G = Generator().to(device)
    D = Discriminator().to(device)
    G.apply(weights_init)
    D.apply(weights_init)

    print(G)
    print(D)
    total_params_G = sum(p.numel() for p in G.parameters())
    total_params_D = sum(p.numel() for p in D.parameters())
    print(f"Generator params:     {total_params_G:,}")
    print(f"Discriminator params: {total_params_D:,}")

    # ── Optimisers (DCGAN paper: Adam, lr=0.0002, β1=0.5) ──
    opt_G = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, BETA2))
    opt_D = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, BETA2))

    # ── Fixed noise for visual progress tracking ──
    fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)

    G_losses, D_losses = [], []

    print("\n─── Training started ───\n")
    for epoch in range(1, args.epochs + 1):
        epoch_start = time.time()
        G_loss_epoch, D_loss_epoch = 0.0, 0.0

        for i, (real_imgs, _) in enumerate(dataloader):
            real_imgs = real_imgs.to(device)
            B = real_imgs.size(0)

            # ════════════════════════════════════════
            #  Step 1: Train Discriminator
            #  max  log D(x) + log(1 - D(G(z)))
            # ════════════════════════════════════════
            D.zero_grad()

            # Real images
            out_real = D(real_imgs)
            loss_D_real = criterion(out_real, real_label(B, device))

            # Fake images
            noise = torch.randn(B, LATENT_DIM, 1, 1, device=device)
            fake_imgs = G(noise)
            out_fake = D(fake_imgs.detach())   # detach: don't update G here
            loss_D_fake = criterion(out_fake, fake_label(B, device))

            loss_D = loss_D_real + loss_D_fake
            loss_D.backward()
            opt_D.step()

            # ════════════════════════════════════════
            #  Step 2: Train Generator
            #  max  log D(G(z))   [non-saturating loss]
            # ════════════════════════════════════════
            G.zero_grad()

            # Re-evaluate fake images through updated D
            out_fake2 = D(fake_imgs)
            # Label fake as real → Generator wants D to be fooled
            loss_G = criterion(out_fake2, real_label(B, device, smoothing=1.0))
            loss_G.backward()
            opt_G.step()

            G_loss_epoch += loss_G.item()
            D_loss_epoch += loss_D.item()

            if (i + 1) % 100 == 0:
                print(
                    f"  Epoch [{epoch:>3}/{args.epochs}] "
                    f"Step [{i+1:>4}/{len(dataloader)}]  "
                    f"D_loss: {loss_D.item():.4f}  "
                    f"G_loss: {loss_G.item():.4f}"
                )

        # ── End-of-epoch bookkeeping ──
        avg_G = G_loss_epoch / len(dataloader)
        avg_D = D_loss_epoch / len(dataloader)
        G_losses.append(avg_G)
        D_losses.append(avg_D)

        elapsed = time.time() - epoch_start
        print(
            f"Epoch {epoch:>3}/{args.epochs}  "
            f"D_loss_avg: {avg_D:.4f}  G_loss_avg: {avg_G:.4f}  "
            f"({elapsed:.1f}s)"
        )

        # ── Save sample grid every SAVE_INTERVAL epochs ──
        if epoch % SAVE_INTERVAL == 0 or epoch == 1:
            with torch.no_grad():
                samples = G(fixed_noise)
            grid_path = os.path.join(OUTPUT_DIR, f"samples_epoch_{epoch:03d}.png")
            save_image(samples, grid_path, nrow=8, normalize=True)
            print(f"  → Saved sample grid: {grid_path}")

    # ── Save final model weights ──
    torch.save(G.state_dict(), os.path.join(OUTPUT_DIR, "generator.pth"))
    torch.save(D.state_dict(), os.path.join(OUTPUT_DIR, "discriminator.pth"))
    print(f"\nModels saved to ./" + OUTPUT_DIR + "/")

    # ── Plot loss curves ──
    plot_losses(G_losses, D_losses)

    # ── Final generation: 10 sample digits ──
    generate_final_samples(G, device, n=10)


# ─────────────────────────────────────────────
#  Plot Loss Curves
# ─────────────────────────────────────────────
def plot_losses(G_losses, D_losses):
    plt.figure(figsize=(10, 5))
    plt.title("Generator vs Discriminator Loss During Training")
    plt.plot(G_losses, label="Generator",     color="#e63946", linewidth=1.8)
    plt.plot(D_losses, label="Discriminator", color="#457b9d", linewidth=1.8)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "loss_curve.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Loss curve saved: {path}")


# ─────────────────────────────────────────────
#  Generate & Display 10 Digit Samples
# ─────────────────────────────────────────────
def generate_final_samples(G, device, n=10):
    G.eval()
    with torch.no_grad():
        noise = torch.randn(n, LATENT_DIM, 1, 1, device=device)
        imgs = G(noise)                        # (n, 1, 64, 64) in [-1,1]
        imgs = (imgs + 1) / 2                  # → [0, 1] for display

    fig, axes = plt.subplots(1, n, figsize=(n * 2, 2.5))
    fig.suptitle("Generated Handwritten Digits (DCGAN)", fontsize=13)
    for idx, ax in enumerate(axes):
        ax.imshow(imgs[idx, 0].cpu().numpy(), cmap="gray")
        ax.axis("off")
        ax.set_title(f"#{idx+1}", fontsize=9)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "final_10_digits.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Final 10-digit sample saved: {path}")


# ─────────────────────────────────────────────
#  Inference-only mode (--generate flag)
# ─────────────────────────────────────────────
def generate_only():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    G = Generator().to(device)
    ckpt = os.path.join(OUTPUT_DIR, "generator.pth")
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f"No checkpoint at {ckpt}. Train first.")
    G.load_state_dict(torch.load(ckpt, map_location=device))
    print(f"Loaded Generator from {ckpt}")
    generate_final_samples(G, device, n=10)


# ─────────────────────────────────────────────
#  Entry Point
# ─────────────────────────────────────────────
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="DCGAN — MNIST Digit Generation")
    parser.add_argument("--epochs",   type=int,  default=NUM_EPOCHS,
                        help=f"Number of training epochs (default: {NUM_EPOCHS})")
    parser.add_argument("--generate", action="store_true",
                        help="Load saved model and generate samples (skip training)")
    args, unknown = parser.parse_known_args()

    if args.generate:
        generate_only()
    else:
        train(args)


[Device] cuda
Generator(
  (net): Sequential(
    (0): ConvTranspose2d(100, 512, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (10): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU(inplace=True)
    (12): ConvTranspose2d(64, 1, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (13): 